# 9.11 · 训练技巧 / Training Tricks

> **课程定位 / Where this fits**
> 第 11 课，**Part 9 · 深度学习基础**。
> Lesson 11, **Part 9 · Deep Learning Foundations**.
>
> 前几节讲了优化器、调度、初始化、正则化等"零件"。这一节把它们**串成一套可靠的实战训练流程**，并补上工程上让训练**又快又稳**的关键技巧：**梯度裁剪、梯度累积、混合精度、checkpoint、过拟合一个 batch、监控**。这些是从"能跑"到"工程上能用"的分水岭。
> Earlier lessons covered the "parts" — optimizers, schedulers, init, regularization. This one **chains them into a reliable practical training workflow** and adds engineering tricks that make training **fast and stable**: **gradient clipping, gradient accumulation, mixed precision, checkpointing, overfit-one-batch, monitoring**. These separate "it runs" from "production-ready."
>
> 💼 **实战/面试视角**："梯度爆炸怎么办 / 显存不够怎么办 / 怎么 debug 训练不收敛" 都是工程岗高频。
> 💼 **Practical/interview angle:** "what if gradients explode / out of GPU memory / how to debug non-convergence" — frequent for engineering roles.

> 📐 **符号约定 / Notation**
> - $\|g\|$ —— 梯度的范数(大小) / gradient norm (magnitude)

> 💡 **面试相关 / Interview-relevant**
> - "梯度裁剪解决什么"（出镜率 ★★★★，RNN/Transformer 爆炸）
> - "显存不够的几种办法"（★★★★，累积/混合精度/小batch）
> - "梯度累积怎么模拟大 batch"（★★★）
> - "混合精度训练原理"（★★★）
> - "怎么 debug 训练不收敛"（★★★★，过拟合一个batch）

---

## 学习目标 / Learning Objectives
1. 写出一个完整、规范的训练循环（含验证、早停、checkpoint）。
   Write a complete, clean training loop (val, early stopping, checkpoint).
2. 用 **梯度裁剪** 防止梯度爆炸。
   Use gradient clipping to prevent exploding gradients.
3. 用 **梯度累积** 在小显存上模拟大 batch。
   Use gradient accumulation to simulate large batches on small memory.
4. 了解 **混合精度** 如何省显存、提速。
   Understand how mixed precision saves memory and speeds up.
5. 掌握 **过拟合一个 batch** 等调试技巧。
   Master the overfit-one-batch debugging trick.

## 目录 / TOC
1. [一个完整的训练循环 ⭐](#1)
2. [梯度裁剪 ⭐](#2)
3. [梯度累积：小显存模拟大 batch ⭐](#3)
4. [混合精度训练 ⭐](#4)
5. [调试技巧 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 一个完整的训练循环 ⭐ / A Complete Training Loop

把 9.3 的 5 步循环升级成**实战版**：加上 mini-batch、验证集评估、早停、保存最佳权重(checkpoint)。这是工业界训练脚本的骨架——记住这个结构，面试手写训练循环就靠它。
Upgrade the 5-step loop from 9.3 into a **production version**: mini-batches, validation, early stopping, best-weight checkpoint. This is the skeleton of industrial training scripts — memorize it for "write a training loop" interview questions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data/16.0, digits.target, test_size=0.3,
                                          stratify=digits.target, random_state=0)
def to_loader(X, y, bs=64, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)
train_loader = to_loader(X_tr, y_tr, shuffle=True); val_loader = to_loader(X_te, y_te)

def make_net():
    return nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128,10))

torch.manual_seed(0)
net = make_net(); opt = torch.optim.Adam(net.parameters(), lr=1e-3); ce = nn.CrossEntropyLoss()
best_val, best_state, patience, wait = float("inf"), None, 8, 0
tr_hist, val_hist = [], []
for epoch in range(100):
    net.train()                                            # 训练模式(开启 Dropout) / train mode
    for xb, yb in train_loader:                            # 遍历 mini-batch / iterate mini-batches
        opt.zero_grad(); loss = ce(net(xb), yb); loss.backward(); opt.step()  # 标准5步 / standard 5 steps
    net.eval()                                             # 评估模式(关闭 Dropout) / eval mode
    with torch.no_grad():                                  # 验证不需要梯度 / no grad for validation
        vl = np.mean([ce(net(xb), yb).item() for xb, yb in val_loader])
    tr_hist.append(loss.item()); val_hist.append(vl)
    if vl < best_val - 1e-4:                               # 验证改善 → 保存最佳权重(checkpoint) / improved → checkpoint
        best_val, best_state, wait = vl, copy.deepcopy(net.state_dict()), 0
    else:
        wait += 1                                          # 没改善 → 计数 / no improvement → count
        if wait >= patience: print(f"早停于 epoch {epoch} (验证{patience}轮没改善)"); break
net.load_state_dict(best_state)                            # 恢复最佳权重 / restore best weights
acc = (net(torch.tensor(X_te, dtype=torch.float32)).argmax(1) == torch.tensor(y_te)).float().mean()
print(f"最佳验证损失 = {best_val:.3f}, 恢复最佳权重后 test 准确率 = {acc:.3f}")
print("骨架: train()→遍历batch→5步→eval()→验证→checkpoint最佳→早停→恢复最佳权重")


<a id="2"></a>
## 2. 梯度裁剪 ⭐ / Gradient Clipping

有些网络（尤其 **RNN/LSTM、Transformer**）训练时偶尔会出现**梯度爆炸**：某一步梯度突然变得极大，把权重一下子推到很糟糕的地方，损失瞬间变成 NaN。
Some nets (especially **RNN/LSTM, Transformers**) occasionally suffer **exploding gradients**: a step's gradient suddenly becomes huge, throwing weights far off and turning loss into NaN.

**梯度裁剪(gradient clipping)** 是个简单可靠的护栏：如果梯度的总范数 $\|g\|$ 超过阈值 $c$，就**等比例缩小**到 $c$（方向不变，只是限制大小）。一行 `clip_grad_norm_` 就能加上。
**Gradient clipping** is a simple, reliable guardrail: if the total gradient norm $\|g\|$ exceeds threshold $c$, **scale it down proportionally** to $c$ (same direction, capped magnitude). One line: `clip_grad_norm_`.


In [ ]:
# 演示裁剪如何把过大的梯度范数限制在阈值内 / show clipping caps the gradient norm
torch.manual_seed(0)
net = make_net()
xb = torch.tensor(X_tr[:64], dtype=torch.float32); yb = torch.tensor(y_tr[:64])
loss = ce(net(xb), yb) * 1000                              # 人为放大损失制造大梯度 / inflate loss → big grads
loss.backward()
total_norm_before = torch.norm(torch.stack([p.grad.norm() for p in net.parameters()]))
clip_value = 1.0
nn.utils.clip_grad_norm_(net.parameters(), max_norm=clip_value)   # 把总范数裁到 ≤1.0 / clip total norm to ≤1.0
total_norm_after = torch.norm(torch.stack([p.grad.norm() for p in net.parameters()]))
print(f"裁剪前梯度总范数 = {total_norm_before:.2f}  (过大, 可能把权重推飞)")
print(f"裁剪后梯度总范数 = {total_norm_after:.2f}  (限制在阈值 {clip_value} 内, 方向不变)")
print("\n用法: loss.backward() 之后、opt.step() 之前调用 clip_grad_norm_")
print("场景: RNN/LSTM/Transformer 防梯度爆炸; 典型阈值 0.5~5")


<a id="3"></a>
## 3. 梯度累积：小显存模拟大 batch ⭐ / Gradient Accumulation

大 batch 通常训练更稳，但 batch 越大越占显存。如果显存装不下大 batch 怎么办？**梯度累积(gradient accumulation)**：把一个大 batch 拆成几个小 batch，**逐个做 backward 让梯度累加，但不立刻 step**；攒够 $k$ 个小 batch 后再 `opt.step()` 一次。效果≈用 $k$ 倍大的 batch。
Large batches train more stably but cost more memory. What if a large batch won't fit? **Gradient accumulation:** split a large batch into several small ones, **backward each so gradients accumulate without stepping**; after $k$ small batches, `opt.step()` once. Effect ≈ a $k×$ larger batch.

关键点：PyTorch 的 `.backward()` 默认**累加**梯度到 `.grad`（这就是为什么平时每步都要 `zero_grad()`）。梯度累积正是**利用**了这个默认行为——攒够了才 zero。
Key: PyTorch's `.backward()` **accumulates** into `.grad` by default (that's why you normally `zero_grad()` each step). Accumulation **exploits** this — only zero after enough steps.


In [ ]:
torch.manual_seed(0)
net = make_net(); opt = torch.optim.SGD(net.parameters(), lr=0.1)
accum_steps = 4                                            # 每 4 个小 batch 才更新一次 / step every 4 micro-batches
small_loader = to_loader(X_tr, y_tr, bs=16, shuffle=True) # 小 batch=16 / micro-batch of 16
for epoch in range(30):                                    # 多跑几轮才能看出效果 / several epochs
    opt.zero_grad()
    for i, (xb, yb) in enumerate(small_loader):
        loss = ce(net(xb), yb) / accum_steps              # 损失除以k: 让累加的梯度等于平均 / divide so accumulated grad = average
        loss.backward()                                   # 梯度累加进 .grad(不清零) / accumulate into .grad
        if (i + 1) % accum_steps == 0:                    # 攒够 k 个 / after k micro-batches
            opt.step(); opt.zero_grad()                   # 才真正更新并清零 / then step and clear
acc = (net(torch.tensor(X_te, dtype=torch.float32)).argmax(1) == torch.tensor(y_te)).float().mean()
print(f"梯度累积(4×16=有效batch64)训练后 test 准确率 = {acc:.3f}")
print("原理: backward 默认累加梯度; 攒够 k 个小batch 才 step+zero_grad → 等效 k 倍大 batch")
print("用途: 显存装不下大 batch 时, 用时间换显存; 注意损失要除以 k")


<a id="4"></a>
## 4. 混合精度训练 ⭐ / Mixed Precision Training

默认权重和计算用 **32 位浮点(FP32)**。**混合精度(mixed precision)** 让大部分计算用 **16 位浮点(FP16/BF16)**：显存占用**减半**、在支持的 GPU 上**速度大幅提升**（Tensor Core），同时保留 FP32 主权重保证精度。
By default weights/compute use **32-bit float (FP32)**. **Mixed precision** runs most compute in **16-bit (FP16/BF16)**: **halves memory**, **big speedups** on supported GPUs (Tensor Cores), while keeping an FP32 master copy for accuracy.

工程上用 `torch.cuda.amp.autocast`（自动选精度）+ `GradScaler`（FP16 梯度太小会下溢，先放大损失再还原）。**注意：混合精度的提速和省显存主要在 GPU 上才有意义**；本课在 CPU 上只演示 API 写法。
In practice: `torch.cuda.amp.autocast` (auto-pick precision) + `GradScaler` (tiny FP16 grads underflow, so scale loss up then back). **Note: speed/memory gains are GPU-only**; on CPU we just show the API.


In [ ]:
# 仅演示混合精度的标准写法(CPU 上不会真正加速, 收益在 GPU) / API pattern only (gains are GPU-only)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前设备 device = {device} (混合精度的提速/省显存主要在 cuda 上体现)")

torch.manual_seed(0)
net = make_net().to(device); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
use_amp = (device == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)       # 损失缩放器, CPU 上 enabled=False 即无操作 / loss scaler
xb = torch.tensor(X_tr[:64], dtype=torch.float32).to(device); yb = torch.tensor(y_tr[:64]).to(device)
for _ in range(3):
    opt.zero_grad()
    with torch.autocast(device_type=device, enabled=use_amp):  # 自动用 FP16/BF16 计算 / autocast region
        loss = ce(net(xb), yb)
    scaler.scale(loss).backward()                         # 放大损失再反传(防FP16梯度下溢) / scale loss before backward
    scaler.step(opt)                                      # 还原梯度并更新 / unscale & step
    scaler.update()                                       # 调整缩放因子 / adjust scale factor
print(f"混合精度训练 3 步完成, 最后 loss = {loss.item():.3f}")
print("套路: autocast 区域内前向 → scaler.scale(loss).backward() → scaler.step → scaler.update")
print("收益: 显存减半 + GPU 上提速(Tensor Core); 现代大模型训练标配")


<a id="5"></a>
## 5. 调试技巧 + 小结 ⭐ / Debugging Tricks & Summary

训练不收敛时，按这些技巧排查（面试常问"怎么 debug"）：
When training won't converge, debug with these (interviewers love "how do you debug"):
- **过拟合一个 batch**：拿**一个小 batch** 反复训，正确的模型应该能把它的损失压到≈0。如果连一个 batch 都过拟合不了，说明**模型/loss/数据管道有 bug**（最有用的第一招）。
  **Overfit one batch:** train on **a single batch** repeatedly; a correct model should drive its loss to ≈0. If it can't even overfit one batch, there's a **bug in model/loss/data pipeline** (the most useful first check).
- **检查输入数据**：可视化几个样本和标签，确认没标错、没归一化错。
  **Inspect inputs:** visualize samples and labels; verify no mislabeling/normalization bugs.
- **从小学习率试起 / 看梯度范数 / 看 loss 是不是 NaN**：定位爆炸或不动。
  **Try small LR / watch grad norm / check NaN:** locate explosion or stagnation.
- **先关掉所有正则化**跑通，再逐步加回去。
  **Turn off all regularization** first to get a baseline, then add back gradually.

下面演示"过拟合一个 batch"。
Below we demo "overfit one batch."


In [ ]:
# 过拟合一个 batch: 正确的模型应能把单个 batch 的损失压到≈0 / overfit one batch sanity check
torch.manual_seed(0)
net = make_net(); opt = torch.optim.Adam(net.parameters(), lr=1e-2)
xb = torch.tensor(X_tr[:32], dtype=torch.float32); yb = torch.tensor(y_tr[:32])
losses = []
net.train()
for _ in range(200):
    opt.zero_grad(); loss = ce(net(xb), yb); loss.backward(); opt.step(); losses.append(loss.item())
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(losses, lw=2); ax.set_xlabel("step"); ax.set_ylabel("单个batch的损失")
ax.set_title("过拟合一个 batch: 损失能压到≈0 → 模型/loss/数据管道基本正确")
plt.tight_layout(); plt.show()
print(f"单个 batch 训练 200 步后损失 = {losses[-1]:.4f} (≈0 表示管道正确)")
print("调试第一招: 连一个 batch 都过拟合不了 → 一定有 bug(模型/loss/数据), 别急着调超参")


```
完整训练循环: train()→遍历batch→5步→eval()→验证→checkpoint最佳→早停→恢复最佳权重
梯度裁剪: 总范数超阈值就等比缩小; 防 RNN/Transformer 梯度爆炸; backward后 step前
梯度累积: backward默认累加, 攒k个小batch才step+zero → 等效大batch(损失除以k); 用时间换显存
混合精度: FP16计算+FP32主权重; 显存减半+GPU提速; autocast+GradScaler; 大模型标配
调试: 过拟合一个batch(第一招)/查输入/小LR/看梯度范数/先关正则
```

### 💡 面试速查 / Interview cheat-sheet
1. **梯度裁剪**: 范数超阈值等比缩小, 防爆炸(RNN/Transformer)。
   Grad clipping: scale down if norm exceeds threshold, prevents explosion.
2. **梯度累积**: 攒 k 个小 batch 再 step, 等效大 batch, 省显存。
   Grad accumulation: step after k micro-batches, simulates large batch, saves memory.
3. **混合精度**: FP16 计算省显存提速, 配 GradScaler 防下溢。
   Mixed precision: FP16 saves memory/speeds up, GradScaler prevents underflow.
4. **过拟合一个 batch**: 调试第一招, 不行就是有 bug。
   Overfit one batch: first debug check, failure means a bug.
5. **显存不够**: 小 batch + 梯度累积 + 混合精度 + 梯度检查点。
   Out of memory: small batch + accumulation + mixed precision + gradient checkpointing.

### 下一节 / Next
**9.12 分布式训练**——单卡装不下或太慢时, 用多 GPU 并行。数据并行(DDP)、模型并行、参数服务器等概念让训练扩展到大规模。
**9.12 Distributed Training** — when one GPU is too small or slow, parallelize across GPUs. Data parallel (DDP), model parallel, parameter servers scale training up.
